In [1]:
import os
import numpy as np
import scipy as sp
from src.data import DataLoader
from src.timescales import Timescales

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dataset = 'HCP-YA'
data_loader = DataLoader(dataset=dataset)

Initialized DataLoader for dataset: HCP-YA
Base directory: /mnt/storage_ssd_raid/research_data/HCP_YA
Raw data directory: /mnt/storage_ssd_raid/research_data/HCP_YA/HCP_1200
Derivatives data directory: /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives
Output directory: /mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales
Loaded behavioral data: 1206 subjects


# MEG data

In [3]:
subject_ids = np.loadtxt(os.path.join(data_loader.base_dir, 'subject_ids_meg.txt'), dtype=str)
subject_ids

array(['100307', '102816', '104012', '105923', '106521', '108323',
       '109123', '111514', '112920', '113922', '116524', '116726',
       '125525', '133019', '140117', '146129', '149741', '151526',
       '153732', '154532', '156334', '158136', '162026', '162935',
       '164636', '166438', '169040', '172029', '174841', '175237',
       '175540', '177746', '179245', '181232', '182840', '185442',
       '187547', '189349', '191033', '191437', '191841', '192641',
       '195041', '198653', '200109', '204521', '205119', '212318',
       '212823', '214524', '221319', '223929', '233326', '248339',
       '250427', '255639', '257845', '283543', '287248', '293748',
       '352132', '352738', '353740', '358144', '406836', '433839',
       '500222', '512835', '555348', '559053', '568963', '581450',
       '599671', '601127', '660951', '662551', '665254', '667056',
       '679770', '680957', '706040', '707749', '715950', '725751',
       '735148', '783462', '814649', '825048', '872764', '8771

In [4]:
meg_scans = data_loader.register_meg_scans(
    subject_ids=subject_ids,
    atlas='Schaefer4007',
    rest_runs=[3, 4, 5],
    wm_runs=[6, 7]
)

In [5]:
meg_scans = data_loader.validate_scan_registry()

Validating 475 scans in registry...
  Missing: sub-104012_modality-MEG_tr-0.0019660657059158916_condition-rest_scan-3Restin_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/meg_parcellated/104012/104012_task-3Restin_atlas-Schaefer4007_stat-mean_timeseries.ptseries.npy
  Missing: sub-104012_modality-MEG_tr-0.0019660657059158916_condition-rest_scan-4Restin_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/meg_parcellated/104012/104012_task-4Restin_atlas-Schaefer4007_stat-mean_timeseries.ptseries.npy
  Missing: sub-104012_modality-MEG_tr-0.0019660657059158916_condition-rest_scan-5Restin_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/meg_parcellated/104012/104012_task-5Restin_atlas-Schaefer4007_stat-mean_timeseries.ptseries.npy
  Missing: sub-104012_modality-MEG_tr-0.0019660657059158916_condition-task_scan-6Wrkmem_atlas-Schaefer4007 -> /mnt/storage_ssd_raid/research_data/HCP_YA/derivatives/meg_parcellated/

In [6]:
data_loader.export_scan_registry('scan_registry_meg.csv')
scan_registry = data_loader.list_registered_scans()
scan_registry.head()

,scan_string,subject_id,modality,scan,atlas,condition,tr
0,sub-100307_modality-MEG_tr-0.00196606570591589...,100307,MEG,3Restin,Schaefer4007,rest,0.001966
1,sub-100307_modality-MEG_tr-0.00196606570591589...,100307,MEG,4Restin,Schaefer4007,rest,0.001966
2,sub-100307_modality-MEG_tr-0.00196606570591589...,100307,MEG,5Restin,Schaefer4007,rest,0.001966
3,sub-100307_modality-MEG_tr-0.00196606570591589...,100307,MEG,6Wrkmem,Schaefer4007,task,0.001966
4,sub-100307_modality-MEG_tr-0.00196606570591589...,100307,MEG,7Wrkmem,Schaefer4007,task,0.001966


In [7]:
subject_ids = scan_registry['subject_id'].unique()
n_subs = len(subject_ids)
print(n_subs, subject_ids)

77 ['100307' '102816' '105923' '106521' '108323' '109123' '111514' '112920'
 '113922' '116726' '133019' '140117' '146129' '149741' '156334' '158136'
 '162026' '162935' '164636' '166438' '169040' '172029' '175237' '175540'
 '177746' '185442' '189349' '191033' '191437' '191841' '192641' '195041'
 '198653' '204521' '205119' '212318' '212823' '214524' '223929' '248339'
 '250427' '255639' '257845' '283543' '293748' '352738' '353740' '358144'
 '406836' '433839' '512835' '555348' '568963' '581450' '599671' '601127'
 '660951' '662551' '665254' '667056' '679770' '680957' '706040' '707749'
 '715950' '725751' '735148' '783462' '814649' '825048' '872764' '877168'
 '891667' '898176' '912447' '917255' '990366']


## Process subjects one at a time (memory issues)

In [8]:
freq_windows = [
    None,
    # (0, 5),
    # (0, 6),
    # (0, 7)
]

timescale_measures = [
    'exponential_tau',
    'zero_crossing',
    'auc_zero_crossing'
]

In [9]:
# run_task = False

for run_task in [False, True]:
    if run_task:
        rest_runs = None
        wm_runs = [6, 7]
    else:
        rest_runs = [3, 4, 5]
        wm_runs = None

    for subject_id in subject_ids:
        data_loader = DataLoader(dataset=dataset, verbose=False)

        meg_scans = data_loader.register_meg_scans(
            subject_ids=[subject_id,],
            atlas='Schaefer4007',
            rest_runs=rest_runs,
            wm_runs=wm_runs
        )

        meg_data = data_loader.load_batch(
            modalities=['MEG'],
            nuisance_regression=False,
            verbose=True
        )
        
        timescales_processor = Timescales(dataset=dataset, timeseries_data=meg_data, verbose=False)
        if run_task:
            timescales_processor.regress_task_structure()
            
        for freq_window in freq_windows:
            timescales_processor.compute_autocorrelation_function(use_fft=True, freq_window=freq_window, use_residuals=False, verbose=False)
            for timescale_measure in timescale_measures:
                timescales_processor.compute_timescale_measures(method=timescale_measure, suffix=f'freq-{freq_window}', verbose=False)
            
            if run_task:
                timescales_processor.compute_autocorrelation_function(use_fft=True, freq_window=freq_window, use_residuals=True, verbose=False)
                for timescale_measure in timescale_measures:
                    timescales_processor.compute_timescale_measures(method=timescale_measure, suffix=f'resid_freq-{freq_window}', verbose=False)

Loading 3 scans...
  [1/3] 100307 3Restin

Loaded: 3/3 scans


Computing auc_zero_crossing timescales: 100%|██████████| 3/3 [00:00<00:00, 20.81it/s]


Loading 3 scans...
  [1/3] 102816 3Restin

Loaded: 3/3 scans


Computing auc_zero_crossing timescales: 100%|██████████| 3/3 [00:00<00:00, 21.40it/s]


Loading 3 scans...
  [1/3] 105923 3Restin

Loaded: 3/3 scans


...

Computing auc_zero_crossing timescales: 100%|██████████| 2/2 [00:00<00:00, 14.28it/s]


Loading 2 scans...
  [1/2] 917255 6Wrkmem

Loaded: 2/2 scans


Computing auc_zero_crossing timescales: 100%|██████████| 2/2 [00:00<00:00, 13.56it/s]


Loading 2 scans...
  [1/2] 990366 6Wrkmem

Loaded: 2/2 scans


Computing auc_zero_crossing timescales: 100%|██████████| 2/2 [00:00<00:00, 14.61it/s]
